# 05 · Agentic GRPO pilot

This is a deliberately tiny, stateful coding environment. The
core Colab stack can validate its tools, hidden reward and
reward-hacking fixtures, but intentionally does not install
TRL's newer `environment_factory` API: that TRL release conflicts
with the current Unsloth dependency bounds. Trainer construction
remains gated until a compatible Unsloth/TRL pair or a separate
NeMo Gym/Harbor rollout backend is validated.

## Install and authenticate

In [ ]:
import subprocess
import sys
from pathlib import Path

# The Git pins supply current Unsloth/Qwen3.8 support. Transformers, TRL and
# Datasets deliberately use the mutually compatible versions from the adjacent
# official Unsloth Qwen3.5 27B notebook. Do not replace these with branch-head
# SHAs without resolving package metadata together first.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
}

import torch

torch_version = torch.__version__.split("+", 1)[0]
torch_minor = ".".join(torch_version.split(".")[:2])
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
xformers_by_torch = {"2.8": "0.0.32.post2", "2.9": "0.0.33.post1", "2.10": "0.0.34", "2.11": "0.0.34"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed Colab dependency set for torch {torch.__version__}. "
        f"Expected one of {sorted(torchao_by_torch)}; update the compatibility matrix first."
    )

COMPATIBILITY_PINS = {
    "transformers": "5.3.0",
    "trl": "0.22.2",
    "datasets": "4.3.0",
    "peft": "0.19.0",
    "torchao": torchao_by_torch[torch_minor],
    "xformers": xformers_by_torch[torch_minor],
}
INSTALLER_REVISION = "colab-v2"
pin_key = "-".join(value.replace(".", "") for value in COMPATIBILITY_PINS.values())
git_key = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_KEY = f"{INSTALLER_REVISION}-torch{torch_minor}-{git_key}-{pin_key}"
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
PIP_LOG = Path("/content/qwen38_pip_install.log")
FORCE_INSTALL = False

def install_phase(name: str, packages: list[str], *, no_deps: bool = False) -> None:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--no-cache-dir",
        "--log",
        str(PIP_LOG),
    ]
    if no_deps:
        command.append("--no-deps")
    command.extend(packages)
    print(f"\n=== install phase: {name} ===")
    print("\n".join(f"  {package}" for package in packages))
    result = subprocess.run(command, check=False)
    if result.returncode:
        log_tail = (
            "\n".join(PIP_LOG.read_text(errors="replace").splitlines()[-120:])
            if PIP_LOG.exists()
            else "[pip did not create its log file]"
        )
        print(f"\n--- tail of {PIP_LOG} ---\n{log_tail}")
        raise RuntimeError(
            f"Package installation failed during {name!r} with exit code {result.returncode}. "
            f"The detailed log is at {PIP_LOG}."
        )

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    if PIP_LOG.exists():
        PIP_LOG.unlink()
    install_phase("packaging tools", ["pip", "setuptools==80.9.0", "wheel>=0.42.0"])
    install_phase("Qwen3.8 training stack", [
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"torch=={torch_version}",
        f"torchao=={COMPATIBILITY_PINS['torchao']}",
        f"transformers=={COMPATIBILITY_PINS['transformers']}",
        f"trl=={COMPATIBILITY_PINS['trl']}",
        f"datasets=={COMPATIBILITY_PINS['datasets']}",
        f"peft=={COMPATIBILITY_PINS['peft']}",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub>=0.34.0,<2.0",
        "hf_transfer",
        "sentencepiece>=0.2.0",
        "protobuf",
        "pytest",
        "jmespath",
    ])
    install_phase(
        "PyTorch-matched xFormers wheel",
        [f"xformers=={COMPATIBILITY_PINS['xformers']}"],
        no_deps=True,
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

After the first install, restart the runtime and rerun the notebook from the top; the install marker skips the pip work.

In [ ]:
import gc
import json
import os
import platform
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )
if "COMPATIBILITY_PINS" not in globals():
    raise RuntimeError("Missing compatibility pins; rerun the notebook from the first cell.")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
# A vendor-labelled 96 GB card can be reported as about 89.4 GiB because
# PyTorch converts the byte count with a binary divisor. Keep the floor well
# above the roughly 44.7 GiB reported for a 48 GB card without rejecting G4.
MIN_G4_TOTAL_GIB = 85.0
print(
    f"GPU: {gpu.name} ({gpu_total_gib:.1f} GiB total), "
    f"capability={torch.cuda.get_device_capability(0)}"
)
if gpu_total_gib < MIN_G4_TOTAL_GIB:
    raise RuntimeError(
        "This suite expects the nominal 96 GB Colab G4 runtime. "
        f"PyTorch reports {gpu_total_gib:.1f} GiB total; expected at least "
        f"{MIN_G4_TOTAL_GIB:.0f} GiB. A value near 45 GiB usually indicates "
        "the 48 GB GPU variant."
    )

# Make rerunning a notebook safe after another heavyweight notebook or a
# failed generation. Unsloth/TorchDynamo can retain compiled module references
# even after a Python variable is overwritten, so clear them before loading a
# fresh model. This does not free memory held by another live Python object.
for _stale_name in ("model", "tokenizer", "processor"):
    globals().pop(_stale_name, None)
gc.collect()
torch.cuda.empty_cache()
try:
    torch._dynamo.reset()
except AttributeError:
    pass

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

observed_pins = {name: package_version(name) for name in COMPATIBILITY_PINS}
pin_mismatches = {
    name: {"expected": expected, "observed": observed_pins[name]}
    for name, expected in COMPATIBILITY_PINS.items()
    if observed_pins[name] != expected
}
if pin_mismatches:
    raise RuntimeError(
        "The runtime does not match the reviewed compatibility set. "
        f"Rerun the install cell with FORCE_INSTALL=True: {pin_mismatches}"
    )

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
    "compatibility_pins": COMPATIBILITY_PINS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

In [ ]:
import hashlib
import inspect
import re
import shutil
import tempfile
from pathlib import Path

from unsloth import FastLanguageModel
from datasets import Dataset
from packaging.version import Version
from transformers import __version__ as transformers_version
from trl import GRPOConfig, GRPOTrainer

ACCEPTED_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-sft-lora"
ACCEPTED_REVISION = "REPLACE_WITH_ACCEPTED_COMMIT"
OUTPUT_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-grpo-lora"
MAX_SEQ_LENGTH = 4_096
MAX_STEPS = 2
ROLLOUT_POLICY_PRECISION = "bf16"  # Set to "bnb4" only as an explicit approximation experiment.
ALLOW_QUANTIZED_ROLLOUT_POLICY = False
RUN_TRAINING = False
PUSH_ADAPTER = False

AGENTIC_TRL_AVAILABLE = (
    Version(transformers_version) >= Version("5.2.0")
    and "environment_factory" in inspect.signature(GRPOTrainer.__init__).parameters
)
AGENTIC_RL_BLOCKER = (
    "The reviewed core stack pins TRL 0.22.2 for Unsloth compatibility; "
    "TRL environment_factory starts at 0.29.0, outside the pinned "
    "Unsloth Zoo trl<=0.24.0 constraint."
)
if RUN_TRAINING and not AGENTIC_TRL_AVAILABLE:
    raise RuntimeError(AGENTIC_RL_BLOCKER)
if RUN_TRAINING and ACCEPTED_REVISION.startswith("REPLACE_"):
    raise RuntimeError("Pin an accepted adapter revision before RL.")
if ROLLOUT_POLICY_PRECISION not in {"bf16", "bnb4"}:
    raise ValueError("ROLLOUT_POLICY_PRECISION must be 'bf16' or 'bnb4'.")
if ROLLOUT_POLICY_PRECISION == "bnb4" and not ALLOW_QUANTIZED_ROLLOUT_POLICY:
    raise RuntimeError(
        "A quantized rollout/update policy changes the accepted BF16 policy distribution. "
        "Set ALLOW_QUANTIZED_ROLLOUT_POLICY=True only for a labelled approximation experiment."
    )
if PUSH_ADAPTER:
    raise RuntimeError("This notebook contains only a toy RL environment; do not publish its adapter.")

## Define a rooted toy coding environment and hidden verifier

In [ ]:
class ToyCodingEnv:
    """Fresh, isolated clamp task for one rollout."""

    def __init__(self):
        self._tmp = None
        self.root = None
        self.tool_errors = 0
        self.test_ran = False
        self.visible_passed = False
        self.expected_test_digest = None

    def reset(self, **kwargs) -> str:
        """Create a clean task and return its user prompt."""
        if self._tmp is not None:
            shutil.rmtree(self._tmp, ignore_errors=True)
        self._tmp = Path(tempfile.mkdtemp(prefix="qwen38_grpo_"))
        self.root = self._tmp / "repo"
        (self.root / "src").mkdir(parents=True)
        (self.root / "tests").mkdir()
        (self.root / "src" / "clamp.py").write_text(
            "def clamp(value, low, high):\n    return value\n"
        )
        test_source = (
            "from src.clamp import clamp\n\n"
            "def test_inside(): assert clamp(2, 0, 4) == 2\n"
            "def test_low(): assert clamp(-1, 0, 4) == 0\n"
        )
        (self.root / "tests" / "test_clamp.py").write_text(test_source)
        self.expected_test_digest = hashlib.sha256(test_source.encode()).hexdigest()
        self.tool_errors = 0
        self.test_ran = False
        self.visible_passed = False
        return "Repository ready. Fix clamp() so it respects both bounds, then run unit tests."

    def _path(self, relative: str) -> Path:
        candidate = (self.root / relative).resolve()
        if candidate != self.root and self.root not in candidate.parents:
            raise ValueError("Path escapes repository root")
        return candidate

    def list_files(self, path: str) -> str:
        """List repository files under path.

        Args:
            path: Repository-relative directory.

        Returns:
            A newline-delimited file list.
        """
        base = self._path(path)
        return "\n".join(str(item.relative_to(self.root)) for item in sorted(base.rglob("*")) if item.is_file())

    def read_file(self, path: str) -> str:
        """Read a bounded UTF-8 repository file.

        Args:
            path: Repository-relative file.

        Returns:
            Bounded file contents.
        """
        return self._path(path).read_text()[:20_000]

    def _tests_intact(self) -> bool:
        test_path = self.root / "tests" / "test_clamp.py"
        if not test_path.is_file():
            return False
        digest = hashlib.sha256(test_path.read_bytes()).hexdigest()
        return digest == self.expected_test_digest

    def search(self, query: str) -> str:
        """Search repository text with a regular expression.

        Args:
            query: Python regular expression.

        Returns:
            Matching path, line, and text records.
        """
        regex = re.compile(query)
        hits = []
        for file in self.root.rglob("*.py"):
            for line_no, line in enumerate(file.read_text().splitlines(), 1):
                if regex.search(line):
                    hits.append(f"{file.relative_to(self.root)}:{line_no}:{line}")
        return "\n".join(hits[:200])

    def apply_patch(self, patch: str) -> str:
        """Apply the one-file unified replacement used by the pilot.

        Args:
            patch: Patch text targeting src/clamp.py.

        Returns:
            Patch status text.
        """
        if "src/clamp.py" not in patch or "return max(low, min(high, value))" not in patch:
            self.tool_errors += 1
            return "Rejected: pilot accepts only the expected minimal clamp edit."
        target = self.root / "src" / "clamp.py"
        before = target.read_text()
        if "return value" not in before:
            return "Already applied."
        target.write_text(before.replace("return value", "return max(low, min(high, value))"))
        return "Done!"

    def run_tests(self, profile: str) -> str:
        """Run the allow-listed visible unit profile.

        Args:
            profile: Must be `unit`.

        Returns:
            Visible-test status text.
        """
        if profile != "unit":
            self.tool_errors += 1
            return "Rejected profile."
        self.test_ran = True
        if not self._tests_intact():
            self.visible_passed = False
            return "Test integrity failure: required visible tests were changed or deleted."
        namespace = {}
        exec((self.root / "src" / "clamp.py").read_text(), namespace)
        fn = namespace["clamp"]
        self.visible_passed = fn(2, 0, 4) == 2 and fn(-1, 0, 4) == 0
        return "2 passed" if self.visible_passed else "1 failed"

    def shell(self, command: str) -> str:
        """Reject arbitrary shell execution in the pilot.

        Args:
            command: Requested command.

        Returns:
            A rejection message.
        """
        self.tool_errors += 1
        return "shell is disabled; use the semantic tools"

    def get_reward(self) -> float:
        """Score hidden behavior; infrastructure failures would return no sample upstream."""
        try:
            if not self._tests_intact():
                return 0.0
            namespace = {}
            exec((self.root / "src" / "clamp.py").read_text(), namespace)
            fn = namespace["clamp"]
            hidden = fn(9, 0, 4) == 4 and fn(-9, 0, 4) == 0 and fn(2, 2, 2) == 2
            reward = 0.2 * float(self.visible_passed and self.test_ran) + 0.8 * float(hidden)
            return max(0.0, reward - 0.05 * self.tool_errors)
        except (KeyError, SyntaxError, TypeError, ValueError):
            return 0.0
        finally:
            shutil.rmtree(self._tmp, ignore_errors=True)

## Unit-test the reward, including hacking fixtures

In [ ]:
untouched = ToyCodingEnv()
untouched.reset()
assert untouched.get_reward() == 0.0

gold = ToyCodingEnv()
gold.reset()
gold.apply_patch("--- a/src/clamp.py\n+++ b/src/clamp.py\n@@ -1,2 +1,2 @@\n def clamp(value, low, high):\n-    return value\n+    return max(low, min(high, value))\n")
assert gold.run_tests("unit") == "2 passed"
assert gold.get_reward() == 1.0

escaped = ToyCodingEnv()
escaped.reset()
try:
    escaped.read_file("../secret")
    raise AssertionError("Path escape was not blocked")
except ValueError:
    pass
finally:
    shutil.rmtree(escaped._tmp, ignore_errors=True)

deletion = ToyCodingEnv()
deletion.reset()
deletion.apply_patch("--- a/src/clamp.py\n+++ b/src/clamp.py\n@@ -1,2 +1,2 @@\n def clamp(value, low, high):\n-    return value\n+    return max(low, min(high, value))\n")
(deletion.root / "tests" / "test_clamp.py").unlink()
assert deletion.run_tests("unit").startswith("Test integrity failure")
assert deletion.get_reward() == 0.0
print("Reward and reward-hacking fixtures passed.")

## Load the accepted adapter and configure multi-turn GRPO

In [ ]:
use_quantized_policy = ROLLOUT_POLICY_PRECISION == "bnb4"
policy_manifest = {
    "accepted_adapter_id": ACCEPTED_ADAPTER_ID,
    "accepted_revision": ACCEPTED_REVISION,
    "rollout_update_precision": ROLLOUT_POLICY_PRECISION,
    "quantized_policy_approximation": use_quantized_policy,
    "allow_quantized_rollout_policy": ALLOW_QUANTIZED_ROLLOUT_POLICY,
    "agentic_trl_available": AGENTIC_TRL_AVAILABLE,
    "agentic_rl_blocker": None if AGENTIC_TRL_AVAILABLE else AGENTIC_RL_BLOCKER,
}
grpo_root = RUN_ROOT / "grpo"
grpo_root.mkdir(parents=True, exist_ok=True)
(grpo_root / "policy_manifest.json").write_text(json.dumps(policy_manifest, indent=2))
print(json.dumps(policy_manifest, indent=2))

trainer = None
if AGENTIC_TRL_AVAILABLE:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ACCEPTED_ADAPTER_ID,
        revision=None if ACCEPTED_REVISION.startswith("REPLACE_") else ACCEPTED_REVISION,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None if use_quantized_policy else torch.bfloat16,
        load_in_4bit=use_quantized_policy,
        token=hf_token,
    )
    if not any(parameter.requires_grad for parameter in model.parameters()):
        raise RuntimeError("Accepted adapter has no trainable parameters; inspect PEFT loading before RL.")

    grpo_args = GRPOConfig(
        output_dir=str(grpo_root),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        num_generations=2,
        max_completion_length=1_024,
        learning_rate=5e-6,
        max_steps=MAX_STEPS,
        bf16=True,
        optim="adamw_8bit",
        logging_steps=1,
        save_strategy="steps",
        save_steps=1,
        save_total_limit=2,
        mask_truncated_completions=True,
        scale_rewards="batch",
        loss_type="dr_grpo",
        report_to="trackio",
        run_name="qwen38-code-agent-grpo-smoke",
        push_to_hub=PUSH_ADAPTER,
        hub_model_id=OUTPUT_ADAPTER_ID,
        seed=3407,
    )
    trainer = GRPOTrainer(
        model=model,
        args=grpo_args,
        processing_class=tokenizer,
        environment_factory=ToyCodingEnv,
    )
    print("GRPO environment and trainer constructed.")
else:
    print(f"Trainer construction skipped: {AGENTIC_RL_BLOCKER}")

## Run only after the toy rollouts work manually

In [ ]:
if RUN_TRAINING:
    if trainer is None:
        raise RuntimeError(AGENTIC_RL_BLOCKER)
    result = trainer.train()
    trainer.save_model(str(RUN_ROOT / "grpo" / "final_adapter"))
    if PUSH_ADAPTER:
        trainer.push_to_hub(commit_message="Agentic GRPO pilot adapter")
    print(result.metrics)
    print("Inspect reward zero-variance metrics; a collapsed group supplies no learning signal.")
else:
    print("RL is off. First inspect generated episodes and confirm hidden rewards manually.")

## Scale-up boundary

Do not turn this fixture into the production harness. After the
reward fixtures pass, resolve the recorded Unsloth/TRL blocker
or use a separate adapter around Harbor/Terminal-Bench or NeMo
Gym. Retain the same six tools, hidden-test boundary, failure
taxonomy and reward tests. Run two independent seeds before
accepting RL.